# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is described using a [Croissant schema](https://mlcommons.org/croissant/) accessible via a URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

### List all record sets and their fields

In [ ]:
# List all record sets, their @id and field ids
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s) in this dataset.\n")
for rec in record_sets:
    print(f"RecordSet: {rec.id}")
    print(f"  Name: {rec.name}")
    print(f"  Description: {rec.description}")
    field_ids = [f.id for f in rec.fields]
    print(f"  Fields (@id): {field_ids}\n")

### List the first record of each record set for a quick look
Below, we print the first record of each main record set.

In [ ]:
for rec in record_sets:
    print(f"\nFirst record of RecordSet {rec.id}:")
    try:
        records = dataset.records(record_set=rec.id)
        first_record = next(records)
        print(first_record)
    except StopIteration:
        print("(No records found)")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Use the record set and field `@id`s from above.

In [ ]:
# Extract all records from each record set
dataframes = {}
for rec in record_sets:
    rec_id = rec.id
    print(f"Reading records from RecordSet: {rec_id}")
    df = pd.DataFrame(dataset.records(record_set=rec_id))
    dataframes[rec_id] = df
    print(f"  Columns: {df.columns.tolist()}\n  #Rows: {len(df)}\n")
# For demo, use the first record set
if len(record_sets) > 0:
    main_record_set_id = record_sets[0].id
    print(f"Preview records from main RecordSet: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

In [ ]:
# Identify candidate numeric fields
main_df = dataframes[main_record_set_id]
numeric_columns = main_df.select_dtypes(include=['int', 'float']).columns.tolist()
print(f"Numeric fields detected: {numeric_columns}")
# Pick the first available numeric field for demonstration
if len(numeric_columns) > 0:
    numeric_field_id = numeric_columns[0]  # By column name, which should match Croissant field @id.
    print(f"Using numeric field: {numeric_field_id}")

    # Filter records where value is above an arbitrary threshold (e.g., 10)
    threshold = 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:\nRows: {len(filtered_df)}")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_colname = f"{numeric_field_id}_normalized"
    filtered_df[norm_colname] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} (z-score):")
    display(filtered_df[[numeric_field_id, norm_colname]].head())

    # Try grouping by a text field, if available
    text_columns = [col for col in main_df.columns if main_df[col].dtype == object and col != numeric_field_id]
    if text_columns:
        group_field = text_columns[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print("Mean of numeric field by group:")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize field distributions or relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_columns) > 0:
    # Histogram of the main numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a second numeric field exists, show scatter
    if len(numeric_columns) > 1:
        second_numeric_field = numeric_columns[1]
        plt.figure(figsize=(7,7))
        sns.scatterplot(data=main_df, x=numeric_field_id, y=second_numeric_field)
        plt.title(f"{numeric_field_id} vs {second_numeric_field}")
        plt.show()
else:
    print("No numeric fields to visualize.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 protein mass spectrometry dataset using `mlcroissant`.

- We accessed the dataset via its Croissant schema, listing available record sets and fields by their `@id`s.
- Records were loaded into pandas DataFrames, filtered, normalized, and grouped for exploratory data analysis.
- Simple visualizations of the numeric fields provided an overview of the data distribution.

For more advanced analysis, you can continue by integrating specific biological hypotheses or extending the EDA and visualization sections.